In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import matplotlib.pyplot as plt
import numpy as np
import torch
from glob import glob
from imrw import imread

from src.inference.predictor import Predictor

In [ ]:
RUN_DIR = "run/20260422_143719"  # set directly — adjust to your run
device = "cuda" if torch.cuda.is_available() else "cpu"
run_dir = str(REPO_ROOT / RUN_DIR)
predictor = Predictor(run_dir=run_dir, device=device)

In [ ]:
files = glob(str(REPO_ROOT / "data" / "default" / "2d" / "*.png"))
file = files[0]

image = imread(file)
image = image[:64, :64, 0]

anchor_images = [image]
anchor_indices = [8]

raw = predictor.predict(
    anchor_images=anchor_images,
    anchor_indices=anchor_indices,
    seed=0,
)
volume = raw[..., 0] if predictor.in_channels == 1 else raw

In [ ]:
def to_display(img):
    return np.clip((img + 1.0) / 2.0, 0.0, 1.0)


D_axis = volume.shape[0]
indices = sorted(
    {0, D_axis // 4, D_axis // 2, 3 * D_axis // 4, D_axis - 1} | set(anchor_indices)
)

fig, axes = plt.subplots(1, len(indices), figsize=(2 * len(indices), 2.4))
cmap = "gray" if predictor.in_channels == 1 else None
for ax, k in zip(axes, indices):
    ax.imshow(to_display(volume[k]), cmap=cmap, vmin=0.0, vmax=1.0)
    title = f"k={k}" + (" (anchor)" if k in anchor_indices else "")
    ax.set_title(title, fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
fig.suptitle("generated slices along axis 0", y=1.02)
plt.tight_layout()
plt.show()